# 02 — Extraction PDF et OCR

Extraction page par page avec PyMuPDF. Les pages contenant très peu de texte sont signalées pour un éventuel OCR.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
INDEX = DATA / "index"
for directory in (RAW, PROCESSED, INDEX):
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

In [ ]:
import hashlib
import json
import fitz

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def extract_pdf(path: Path) -> list[dict]:
    pages = []
    with fitz.open(path) as document:
        for page_number, page in enumerate(document, start=1):
            text = page.get_text("text").strip()
            pages.append({
                "file": path.name,
                "page": page_number,
                "text": text,
                "needs_ocr": len(text) < 80,
                "sha256": sha256_file(path),
            })
    return pages

pdf_files = sorted(RAW.glob("*.pdf"))
extracted = [page for pdf in pdf_files for page in extract_pdf(pdf)]
output = PROCESSED / "pages.jsonl"
with output.open("w", encoding="utf-8") as stream:
    for row in extracted:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")
{"pdf_count": len(pdf_files), "page_count": len(extracted), "output": str(output)}

Pour les documents scannés, installer l'option `ocr` et Tesseract avec les langues arabe et française. L'OCR doit rester traçable : on conserve toujours le numéro de page et le PDF original.